# Multilingual BERT — bert-base-multilingual-uncased

Part of the M.S. thesis *Hate Speech Classification in Roman Urdu: A Comparative Study on Parameter Efficient Fine-Tuning and Prompt Engineering*.

Fine-tunes **Multilingual BERT** (`bert-base-multilingual-uncased`) for binary hate speech classification of Roman Urdu comments with **QLoRA**: 4-bit NF4 quantisation (bitsandbytes) plus LoRA adapters (r=16, alpha=8, dropout=0.05) via Hugging Face PEFT. A custom `Trainer` uses class-weighted cross-entropy to handle the 82/18 class imbalance. The notebook first measures the untrained baseline, then trains for 2 epochs and reports accuracy, balanced accuracy and macro precision/recall/F1 on the held-out test set.

**Setup:** install dependencies with `pip install -r requirements.txt` from the repository root and place the dataset in `data/` (see the main README).

**Hardware:** a CUDA GPU with at least 16 GB of memory is recommended.

In [ ]:
import os

# Paths are relative to this notebook. See data/README.md for the dataset.
DATA_PATH = "../../data/PURRUTT.xlsx"
OUTPUT_DIR = "../../outputs/mbert_qlora"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Weights & Biases logging is off by default; set to "false" to enable it.
os.environ.setdefault("WANDB_DISABLED", "true")
# Gated models (e.g. Llama 3, Gemma) need a Hugging Face token:
# run `huggingface-cli login` or set the HF_TOKEN environment variable before starting.

In [ ]:
import os
import random
import functools
import csv
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
import evaluate

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, confusion_matrix, classification_report, balanced_accuracy_score, accuracy_score

from datasets import Dataset, DatasetDict
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model
from trl import SFTTrainer
import pandas as pd
import numpy as np
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    GemmaTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

In [ ]:
df = pd.read_excel(DATA_PATH)
df

In [ ]:
train_end_point = int(df.shape[0]*0.6)
val_end_point = int(df.shape[0]*0.8)
df_train = df.iloc[:train_end_point,:]
df_val = df.iloc[train_end_point:val_end_point,:]
df_test = df.iloc[val_end_point:,:]
print(df_train.shape, df_test.shape, df_val.shape)

In [ ]:
# Converting pandas DataFrames into Hugging Face Dataset objects:
dataset_train = Dataset.from_pandas(df_train.drop('Urdu',axis=1))
dataset_val = Dataset.from_pandas(df_val.drop('Urdu',axis=1))
dataset_test = Dataset.from_pandas(df_test.drop('Urdu',axis=1))

In [ ]:
# Shuffle the training dataset
dataset_train_shuffled = dataset_train.shuffle(seed=42)  # Using a seed for reproducibility

In [ ]:
# Combine them into a single DatasetDict
dataset = DatasetDict({
    'train': dataset_train_shuffled,
    'val': dataset_val,
    'test': dataset_test
})
dataset

In [ ]:
class_weights=(1/df_train.Toxic.value_counts(normalize=True).sort_index()).tolist()
class_weights=torch.tensor(class_weights)
class_weights=class_weights/class_weights.sum()
class_weights

### Prerequisites
* nf4(4-bit NormalFloat(NF4)) : https://www.kaggle.com/code/lorentzyeung/what-s-4-bit-quantization-how-does-it-help-llama2

In [ ]:
model_id = "bert-base-multilingual-uncased"
quantization_config = BitsAndBytesConfig(
    load_in_4bit = True, # enable 4-bit quantization
    bnb_4bit_quant_type = 'nf4', # information theoretically optimal dtype for normally distributed weights
    bnb_4bit_use_double_quant = True, # quantize quantized weights //insert xzibit meme
    bnb_4bit_compute_dtype = torch.bfloat16 # optimized fp format for ML
)

In [ ]:
lora_config = LoraConfig(
    r = 16, # the dimension of the low-rank matrices
    lora_alpha = 8, # scaling factor for LoRA activations vs pre-trained weight activations
    # target_modules = ['q_proj', 'k_proj', 'v_proj', 'o_proj'],
    lora_dropout = 0.05, # dropout probability of the LoRA layers
    bias = 'none', # wether to train bias weights, set to 'none' for attention layers
    task_type = 'SEQ_CLS',
    inference_mode=False,        # Enable training mode

)

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    # Change num_labels to match the original model's output size (5)
    num_labels=2
)




model

In [ ]:
print(model.config)

In [ ]:
model = prepare_model_for_kbit_training(model)
model

In [ ]:
model = get_peft_model(model, lora_config)
model

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_id, add_prefix_space=True)

tokenizer.pad_token_id = tokenizer.eos_token_id
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
model.config.pad_token_id = tokenizer.pad_token_id
model.config.use_cache = False
model.config.pretraining_tp = 1

In [ ]:
sentences = df_test.Comment.tolist()
sentences[0:2]

In [ ]:
from tqdm import tqdm
import torch

# Convert summaries to a list
sentences = df_test.Comment.tolist()

# Define the batch size
batch_size = 50  # You can adjust this based on your system's memory capacity

# Initialize an empty list to store the model outputs
all_outputs = []

# Ensure tokenizer has a padding token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token  # Assign EOS as PAD if available
    if tokenizer.pad_token is None:
        tokenizer.add_special_tokens({'pad_token': '[PAD]'})  # Manually add PAD token

# Process the sentences in batches with tqdm progress bar
num_batches = len(sentences) // batch_size + (len(sentences) % batch_size != 0)  # Total batches

for i in tqdm(range(0, len(sentences), batch_size), desc="Processing Batches", unit="batch", total=num_batches):
    # Get the batch of sentences
    batch_sentences = sentences[i:i + batch_size]

    # Tokenize the batch
    inputs = tokenizer(batch_sentences, return_tensors="pt", padding=True, truncation=True, max_length=128)

    # Move tensors to the device where the model is (e.g., GPU or CPU)
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    inputs = {k: v.to(device) for k, v in inputs.items()}

    # Perform inference and store the logits
    with torch.no_grad():
        outputs = model(**inputs)
        all_outputs.append(outputs['logits'])

print("✅ Processing complete!")

In [ ]:
final_outputs = torch.cat(all_outputs, dim=0)
final_outputs

In [ ]:
final_outputs.argmax(axis=1)

In [ ]:
df_test['predictions']=final_outputs.argmax(axis=1).cpu().numpy()
df_test['predictions']

In [ ]:
df_test['predictions'].value_counts()

In [ ]:
def get_performance_metrics(df_test):
  y_test = df_test.Toxic
  y_pred = df_test.predictions

  print("Confusion Matrix:")
  print(confusion_matrix(y_test, y_pred))

  print("\nClassification Report:")
  print(classification_report(y_test, y_pred))

  print("Balanced Accuracy Score:", balanced_accuracy_score(y_test, y_pred))
  print("Accuracy Score:", accuracy_score(y_test, y_pred))

In [ ]:
get_performance_metrics(df_test)

In [ ]:
MAX_LEN = 128
col_to_delete = ['Urdu']

def bert_preprocessing_function(examples):
    return tokenizer(examples['Comment'], truncation=True, max_length=MAX_LEN)

tokenized_datasets = dataset.map(bert_preprocessing_function, batched=True)
tokenized_datasets = tokenized_datasets.rename_column("Toxic", "label")
tokenized_datasets.set_format("torch")

In [ ]:
collate_fn = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return {'balanced_accuracy' : balanced_accuracy_score(predictions, labels),'accuracy':accuracy_score(predictions,labels)}

In [ ]:
class CustomTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        # Ensure label_weights is a tensor
        if class_weights is not None:
            self.class_weights = torch.tensor(class_weights, dtype=torch.float32).to(self.args.device)
        else:
            self.class_weights = None

    def compute_loss(self, model, inputs, return_outputs=False):
        # Extract labels and convert them to long type for cross_entropy
        labels = inputs.pop("labels").long()

        # Forward pass
        outputs = model(**inputs)

        # Extract logits assuming they are directly outputted by the model
        logits = outputs.get('logits')

        # Compute custom loss with class weights for imbalanced data handling
        if self.class_weights is not None:
            loss = F.cross_entropy(logits, labels, weight=self.class_weights)
        else:
            loss = F.cross_entropy(logits, labels)

        return (loss, outputs) if return_outputs else loss

In [ ]:
training_args = TrainingArguments(
    output_dir = os.path.join(OUTPUT_DIR, "checkpoints"),
    learning_rate = 1e-4,
    per_device_train_batch_size = 8,
    per_device_eval_batch_size = 8,
    num_train_epochs = 2,
    weight_decay = 0.01,
    evaluation_strategy = 'epoch',
    save_strategy = 'epoch',
    load_best_model_at_end = True
)

In [ ]:
trainer = CustomTrainer(
    model = model,
    args = training_args,
    train_dataset = tokenized_datasets['train'],
    eval_dataset = tokenized_datasets['val'],
    tokenizer = tokenizer,
    data_collator = collate_fn,
    compute_metrics = compute_metrics,
    class_weights=class_weights,
)

In [ ]:
train_result = trainer.train()

In [ ]:
from tqdm import tqdm
import torch

def make_predictions(model, df_test):
    # Convert summaries to a list
    sentences = df_test.Comment.tolist()

    # Define the batch size
    batch_size = 32  # You can adjust this based on your system's memory capacity

    # Initialize an empty list to store the model outputs
    all_outputs = []

    # Process the sentences in batches with tqdm progress bar
    for i in tqdm(range(0, len(sentences), batch_size), desc="Processing Batches", unit="batch"):
        # Get the batch of sentences
        batch_sentences = sentences[i:i + batch_size]

        # Tokenize the batch
        inputs = tokenizer(batch_sentences, return_tensors="pt", padding=True, truncation=True, max_length=128)

        # Move tensors to the device where the model is (e.g., GPU or CPU)
        device = 'cuda' if torch.cuda.is_available() else 'cpu'
        inputs = {k: v.to(device) for k, v in inputs.items()}

        # Perform inference and store the logits
        with torch.no_grad():
            outputs = model(**inputs)
            all_outputs.append(outputs['logits'])

    # Concatenate all outputs
    final_outputs = torch.cat(all_outputs, dim=0)

    # Convert logits to class predictions
    df_test['predictions'] = final_outputs.argmax(axis=1).cpu().numpy()

    # Optional: Map predictions to categories if needed
    # df_test['predictions'] = df_test['predictions'].apply(lambda l: category_map[l])

    print("Prediction process completed!")

make_predictions(model, df_test)

In [ ]:
def get_performance_metrics(df_test):
  y_test = df_test.Toxic
  y_pred = df_test.predictions

  print("Confusion Matrix:")
  print(confusion_matrix(y_test, y_pred))

  print("\nClassification Report:")
  print(classification_report(y_test, y_pred))

  print("Balanced Accuracy Score:", balanced_accuracy_score(y_test, y_pred))
  print("Accuracy Score:", accuracy_score(y_test, y_pred))

In [ ]:
get_performance_metrics(df_test)

In [ ]:
from sklearn.metrics import precision_recall_fscore_support

def calculate_macro_scores(y_true, y_pred):
    y_test = df_test.Toxic
    y_pred = df_test.predictions
    precision, recall, f1, _ = precision_recall_fscore_support(y_test, y_pred, average='macro')

    print(f"Macro Precision: {precision:.4f}")
    print(f"Macro Recall: {recall:.4f}")
    print(f"Macro F1-score: {f1:.4f}")

    return precision, recall, f1


y_test = df_test.Toxic
y_pred = df_test.predictions
calculate_macro_scores(y_test, y_pred)

In [ ]:
metrics = train_result.metrics
max_train_samples = len(dataset_train)
metrics["train_samples"] = min(max_train_samples, len(dataset_train))
trainer.log_metrics("train", metrics)
trainer.save_metrics("train", metrics)
trainer.save_state()

In [ ]:
trainer.save_model(os.path.join(OUTPUT_DIR, "adapter"))